In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("GALAXY_DATASET/final_15000.csv")
df.head()

In [ ]:
# 10 images are missing from the folder, drop those rows
img_folder = "Images Differing sizes/224"
valid = set(int(f.replace(".jpg", "")) for f in os.listdir(img_folder) if f.endswith(".jpg"))
df = df[df["asset_id"].isin(valid)].reset_index(drop=True)
print(len(df))  # should be 14990

In [ ]:
def make_label(row, thresh=0.5):
    smooth = row["t01_smooth_or_features_a01_smooth_debiased"]
    featured = row["t01_smooth_or_features_a02_features_or_disk_debiased"]
    edgeon = row["t02_edgeon_a04_yes_debiased"]
    bar = row["t03_bar_a06_bar_debiased"]
    spiral = row["t04_spiral_a08_spiral_debiased"]

    if smooth >= thresh:
        return "elliptical"
    elif featured >= thresh:
        if edgeon >= thresh:
            return "edge_on"
        elif bar >= thresh and spiral >= thresh:
            return "barred_spiral"
        elif spiral >= thresh:
            return "spiral"
    return None

df["label"] = df.apply(make_label, axis=1)
df = df.dropna(subset=["label"]).reset_index(drop=True)
print(df["label"].value_counts())

In [ ]:
classes = ["elliptical", "spiral", "barred_spiral", "edge_on"]
label2idx = {c: i for i, c in enumerate(classes)}
df["label_idx"] = df["label"].map(label2idx)
label2idx

In [ ]:
class GalaxyDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        path = os.path.join(self.img_dir, f"{int(row['asset_id'])}.jpg")
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(row["label_idx"])

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])
print(f"train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}")

In [ ]:
# imagenet mean/std since we're using pretrained models
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(360),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

In [ ]:
train_ds = GalaxyDataset(train_df, img_folder, transform=train_tfms)
val_ds   = GalaxyDataset(val_df,   img_folder, transform=val_tfms)
test_ds  = GalaxyDataset(test_df,  img_folder, transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

print(f"batches — train: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}")